In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.neighbors import LocalOutlierFactor
from scipy.spatial import KDTree
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

class AdvancedPointCloudDenoiser:
    def __init__(self, method='combined', n_clusters=5, eps=0.5, min_samples=5):
        """
        Advanced noise removal class using combined algorithms
        :param method: noise removal method ('combined', 'dbscan', 'lof', 'statistical')
        :param n_clusters: number of clusters for K-means
        :param eps: epsilon distance for DBSCAN
        :param min_samples: min samples for DBSCAN
        """
        self.method = method
        self.n_clusters = n_clusters
        self.eps = eps
        self.min_samples = min_samples
        self.cleaned_points_ = None
        
    def fit_transform(self, points):
        """Run denoising with the selected method"""
        print(" Starting advanced noise removal...")
        
        if self.method == 'combined':
            return self._combined_method(points)
        elif self.method == 'dbscan':
            return self._dbscan_method(points)
        elif self.method == 'lof':
            return self._lof_method(points)
        elif self.method == 'statistical':
            return self._statistical_method(points)
        else:
            raise ValueError("Invalid method")
    
    def _combined_method(self, points):
        """Combined method: PCA + local density + statistical analysis"""
        print(" Using advanced combined method")
        
        # Stage 1: initial clustering with K-means
        kmeans = KMeans(n_clusters=min(self.n_clusters, len(points)//3), random_state=42)
        labels = kmeans.fit_predict(points)
        
        all_inliers = []
        
        for cluster_id in range(kmeans.n_clusters):
            cluster_points = points[labels == cluster_id]
            
            if len(cluster_points) < 4:
                continue
                
            # Denoise within each cluster
            cleaned_cluster = self._clean_cluster_advanced(cluster_points)
            all_inliers.extend(cleaned_cluster)
        
        self.cleaned_points_ = np.array(all_inliers)
        return self.cleaned_points_
    
    def _clean_cluster_advanced(self, cluster_points):
        """Advanced denoising within a single cluster"""
        if len(cluster_points) < 4:
            return cluster_points
        
        # 1. PCA to find the principal direction
        pca = PCA(n_components=2)
        pca.fit(cluster_points)
        
        # Principal direction vector
        main_direction = pca.components_[0]
        main_control_point = np.mean(cluster_points, axis=0)
        
        # 2. Compute distance from the principal axis
        distances = self._distance_to_line(cluster_points, main_control_point, main_direction)
        
        # 3. Local density analysis
        tree = KDTree(cluster_points)
        densities = []
        for point in cluster_points:
            indices = tree.query_ball_point(point, self.eps)
            densities.append(len(indices))
        
        densities = np.array(densities)
        
        # 4. Combined scoring for outlier detection
        z_scores_dist = np.abs(stats.zscore(distances))
        z_scores_den = np.abs(stats.zscore(densities))
        
        # Combined score (large distance + low density = outlier)
        combined_scores = z_scores_dist * (2 - z_scores_den/np.max(z_scores_den))
        
        # Dynamic threshold based on data distribution
        threshold = np.percentile(combined_scores, 85)  # retain the best 85% of points
        
        inlier_mask = combined_scores <= threshold
        inliers = cluster_points[inlier_mask]
        
        return inliers
    
    def _dbscan_method(self, points):
        """DBSCAN-based density method"""
        print(" Using DBSCAN method")
        
        dbscan = DBSCAN(eps=self.eps, min_samples=self.min_samples)
        labels = dbscan.fit_predict(points)
        
        # points with label -1 are outliers
        inlier_mask = labels != -1
        self.cleaned_points_ = points[inlier_mask]
        
        return self.cleaned_points_
    
    def _lof_method(self, points):
        """Local Outlier Factor method"""
        print(" Using LOF method")
        
        lof = LocalOutlierFactor(n_neighbors=min(20, len(points)-1), contamination=0.1)
        labels = lof.fit_predict(points)
        
        inlier_mask = labels != -1
        self.cleaned_points_ = points[inlier_mask]
        
        return self.cleaned_points_
    
    def _statistical_method(self, points):
        """Advanced statistical method"""
        print(" Using advanced statistical method")
        
        # Compute distance from centroid
        center = np.mean(points, axis=0)
        distances = np.linalg.norm(points - center, axis=1)
        
        # Z-score with dynamic threshold
        z_scores = np.abs(stats.zscore(distances))
        
        # Threshold based on data distribution
        if len(points) > 50:
            threshold = 2.5  # stricter for larger datasets
        else:
            threshold = 2.0  # more lenient for smaller datasets
        
        inlier_mask = z_scores < threshold
        self.cleaned_points_ = points[inlier_mask]
        
        return self.cleaned_points_
    
    def _distance_to_line(self, points, line_point, line_direction):
        """Compute perpendicular distance from each point to a line"""
        line_direction = line_direction / np.linalg.norm(line_direction)
        vectors = points - line_point
        projections = np.dot(vectors, line_direction)
        projected_points = line_point + projections[:, np.newaxis] * line_direction
        distances = np.linalg.norm(points - projected_points, axis=1)
        return distances

    def visualize_results(self, original_points, save_plot=True):
        """Visualize denoising results"""
        if self.cleaned_points_ is None:
            print(" Run denoising first!")
            return
        
        fig = plt.figure(figsize=(20, 10))
        
        # 3D comparison plot
        ax1 = fig.add_subplot(121, projection='3d')
        ax1.scatter(original_points[:, 0], original_points[:, 1], original_points[:, 2], 
                   c='red', alpha=0.6, s=40, label='Removed')
        ax1.scatter(self.cleaned_points_[:, 0], self.cleaned_points_[:, 1], self.cleaned_points_[:, 2], 
                   c='green', alpha=0.8, s=50, label='Retained')
        ax1.set_title('Retained vs Removed Points', fontsize=14, fontweight='bold')
        ax1.set_xlabel('X')
        ax1.set_ylabel('Y')
        ax1.set_zlabel('Z')
        ax1.legend()
        
        # 2D top-down view
        ax2 = fig.add_subplot(122)
        ax2.scatter(original_points[:, 0], original_points[:, 1], 
                   c='red', alpha=0.4, s=30, label='Removed')
        ax2.scatter(self.cleaned_points_[:, 0], self.cleaned_points_[:, 1], 
                   c='green', alpha=0.8, s=40, label='Retained')
        ax2.set_title('XY View - Retained vs Removed', fontsize=14, fontweight='bold')
        ax2.set_xlabel('X')
        ax2.set_ylabel('Y')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.suptitle(f'Denoising Results - Method: {self.method}', fontsize=16, fontweight='bold')
        plt.tight_layout()
        plt.show()
        
        if save_plot:
            plt.savefig(f'advanced_denoising_{self.method}.png', dpi=300, bbox_inches='tight')
            print(f"💾 Plot saved to file 'advanced_denoising_{self.method}.png' saved.")

# Function to test all methods
def test_all_methods(points):
    """Test all methods and compare results."""
    methods = ['combined', 'dbscan', 'lof', 'statistical']
    results = {}
    
    print(" Testing all noise removal methods:")
    print("=" * 50)
    
    for method in methods:
        print(f"\
            Testing method: {method}")
        denoiser = AdvancedPointCloudDenoiser(method=method)
        
        try:
            cleaned = denoiser.fit_transform(points)
            results[method] = {
                'points': cleaned,
                'original_count': len(points),
                'cleaned_count': len(cleaned),
                'removed_count': len(points) - len(cleaned),
                'removal_rate': (len(points) - len(cleaned)) / len(points)
            }
            
            print(f"   input points: {len(points)}")
            print(f"   retained: {len(cleaned)}")
            print(f"   Removed: {len(points) - len(cleaned)}")
            print(f"   removal rate: {results[method]['removal_rate']:.2%}")
            
            # Plot results for each method
            denoiser.visualize_results(points, save_plot=True)
            
        except Exception as e:
            print(f"    Error in method {method}: {e}")
            results[method] = None
    
    return results

# Load data from CSV file
def load_data(file_path):
    """Load data from CSV file"""
    try:
        df = pd.read_csv(file_path)
        if df.shape[1] >= 3:
            points = df.iloc[:, :3].values
            print(f" File '{file_path}' loaded. {len(points)} points found.")
            print("/n================================================/n")
            return points
        else:
            raise ValueError("file must have at least 3 columns")
    except:
        try:
            df = pd.read_csv(file_path, header=None)
            points = df.iloc[:, :3].values
            print(f" File '{file_path}' (no header) loaded. {len(points)} points found.")
            return points
        except Exception as e:
            print(f" error loading file: {e}")
            return None

# Main program execution
if __name__ == "__main__":
    print(" Starting advanced noise removal")
    print("=" * 60)
    
    # Read data
    file_path = "1.csv"
    points = load_data(file_path)
    
    if points is None:
        print(" Using sample data...")
        # Generate sample data with a curve and outlier points
        np.random.seed(42)
        
        # Main curve
        t = np.linspace(0, 4*np.pi, 100)
        curve_points = np.column_stack([
            10 * np.cos(t),
            10 * np.sin(t), 
            2 * t
        ])
        
        # far outlier points
        outliers = np.random.uniform(-20, 20, (30, 3))
        
        points = np.vstack([curve_points, outliers])
        np.random.shuffle(points)
    
    print(f" Input data info:")
    print(f"   point count: {len(points)}")
    print(f"   X range: [{points[:, 0].min():.2f}, {points[:, 0].max():.2f}]")
    print(f"   Y range: [{points[:, 1].min():.2f}, {points[:, 1].max():.2f}]")
    print(f"   Z range: [{points[:, 2].min():.2f}, {points[:, 2].max():.2f}]")
    
    # Test all methods
    results = test_all_methods(points)
    
    # Find best method based on removal rate reasonableness
    print("\ Result analysis:")
    print("=" * 40)
    
    best_method = None
    best_score = -1
    
    for method, result in results.items():
        if result is not None:
            removal_rate = result['removal_rate']
            # score: 10-40% removal rate is ideal
            if 0.1 <= removal_rate <= 0.4:
                score = 1 - abs(removal_rate - 0.25)  # closer to 25% removal is better
            else:
                score = 0
                
            if score > best_score:
                best_score = score
                best_method = method
    
    if best_method:
        print(f" best method: {best_method}")
        print("/n================================================/n")
        
        print(f"   input points: {results[best_method]['original_count']}")
        print(f"   retained: {results[best_method]['cleaned_count']}")
        print(f"   removal rate: {results[best_method]['removal_rate']:.2%}")
        
        # Save results from best method
        best_points = results[best_method]['points']
        output_file = f'best_cleaned_points_{best_method}.csv'
        pd.DataFrame(best_points, columns=['X', 'Y', 'Z']).to_csv(output_file, index=False)
        print(f" Best method results saved to '{output_file}'.")
    else:
        print("Warning: no method produced good results. Falling back to combined method.")
        denoiser = AdvancedPointCloudDenoiser(method='combined')
        best_points = denoiser.fit_transform(points)
        output_file = 'final_cleaned_points.csv'
        pd.DataFrame(best_points, columns=['X', 'Y', 'Z']).to_csv(output_file, index=False)
        print(f" final results saved to '{output_file}' saved.")
    
    print("\ Processing complete!") 

# Rezult 

In [ ]:

import os, sys
import numpy as np
import pandas as pd

# try sklearn for speed/robustness
try:
    from sklearn.cluster import KMeans
    from sklearn.neighbors import NearestNeighbors
    SKLEARN = True
except Exception:
    SKLEARN = False

# optional plotting
try:
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa
    PLOTTING = True
except Exception:
    PLOTTING = False


#  CONFIG (adjust for stricter/gentler denoising)

In [ ]:

INPUT_CSV = '1.csv'  # input file name
OUTPUT_KEPT = 'kept_points_cleaned.csv'
OUTPUT_OUTLIERS = 'outliers_points.csv'
OUTPUT_REPORT = 'per_point_report.csv'

# clustering (K selection)
USER_K = None           # None => auto, or integer to fix k
MAX_K_TRY = 8           # how many k values to try for auto-detection

# per-cluster PCPCA thresholds (robust)
PERP_METHOD = 'mad'     # 'mean_std' or 'mad' (recommended for outlier robustness)
MAD_K = 5.1             # threshold = median + MAD_K * MAD
MEAN_STD_ALPHA = 2    # if using mean_std, smaller => stricter

# density-based rule (local density via k-nearest neighbors)
K_NEIGHBORS = 8         # local neighbors for density
DENSITY_METHOD = 'inv_mean_dist'  # 'inv_mean_dist' or 'median_radius'
DENSITY_PERCENTILE = 10  # if density below this percentile => low-density (30 => lowest 30%)

# Combination rule: a point is removed if
#    (a) perpendicular distance > perp_thresh  AND  density <= density_threshold
# or (optional) if very isolated (very low density) even at moderate distance
COMBINE_RULE = 'both'   # 'both' => both conditions required, 'either' => remove if either is met
ISOLATED_DENSITY_FACTOR = 0.1  # if density < factor * median_density => always remove

# iterative removal 
# False     True
ITERATIVE = True
MAX_ITER = 5

# cluster min points
MIN_CLUSTER_SIZE = 5

VERBOSE = True

## find_input_file

In [ ]:

def find_input_file(name):
    if os.path.exists(name):
        return name
    alt = os.path.join('/mnt/data', name)
    if os.path.exists(alt):
        return alt
    raise FileNotFoundError(f"Input file '{name}' not found in CWD or /mnt/data.")

## load_points

In [ ]:

def load_points(path):
    df = pd.read_csv(path)
    # try to find X,Y,Z columns (case-insensitive)
    cols = [c.strip() for c in df.columns]
    lower = [c.lower() for c in cols]
    if 'x' in lower and 'y' in lower and 'z' in lower:
        ix = lower.index('x'); iy = lower.index('y'); iz = lower.index('z')
        pts = df[[cols[ix], cols[iy], cols[iz]]].to_numpy(dtype=float)
    else:
        # fallback: first 3 numeric columns
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) >= 3:
            pts = df[numeric_cols[:3]].to_numpy(dtype=float)
        else:
            pts = df.iloc[:, :3].to_numpy(dtype=float)
    return pts, df

## pca_principal

In [ ]:
def pca_principal(pts):
    # returns mean (mu) and principal direction (unit vector)
    mu = pts.mean(axis=0)
    if pts.shape[0] <= 1:
        return mu, np.array([1.,0.,0.])
    C = np.cov((pts - mu).T, bias=True)
    eigvals, eigvecs = np.linalg.eigh(C)
    principal = eigvecs[:, np.argmax(eigvals)]
    principal = principal / np.linalg.norm(principal)
    return mu, principal

## perp_distances

In [ ]:

def perp_distances(pts, mu, direction):
    d = []
    dirn = direction / np.linalg.norm(direction)
    for p in pts:
        v = p - mu
        proj = dirn * (v.dot(dirn))
        perp = v - proj
        d.append(np.linalg.norm(perp))
    return np.array(d)

## compute_local_density

In [ ]:

def compute_local_density(points, k=K_NEIGHBORS):
    # returns density_score for each point: higher => denser
    n = points.shape[0]
    if n <= k:
        # trivial: all dense (return high equal scores)
        return np.ones(n)
    if SKLEARN:
        nbrs = NearestNeighbors(n_neighbors=k+1, algorithm='auto').fit(points)
        distances, idx = nbrs.kneighbors(points)
        # drop zero distance to itself (first col)
        mean_dist = distances[:,1:].mean(axis=1)
    else:
        # brute-force distances
        D = np.sqrt(((points[:,None,:] - points[None,:,:])**2).sum(axis=2))
        # set diag to large
        np.fill_diagonal(D, np.inf)
        sorted_d = np.sort(D, axis=1)[:, :k]
        mean_dist = sorted_d.mean(axis=1)
    # density score = 1 / (mean_dist + eps)
    eps = 1e-9
    density = 1.0 / (mean_dist + eps)
    return density

## robust_threshold_by_mad

In [ ]:



def robust_threshold_by_mad(values, k=MAD_K):
    med = np.median(values)
    mad = np.median(np.abs(values - med))
    # if mad == 0 (all values identical) fallback to std
    if mad == 0:
        std = values.std(ddof=0)
        return med + k * std
    return med + k * mad


## detect_outliers_pipeline

In [ ]:

def detect_outliers_pipeline(points,
                             user_k=USER_K,
                             max_k_try=MAX_K_TRY,
                             perp_method=PERP_METHOD,
                             mad_k=MAD_K,
                             mean_std_alpha=MEAN_STD_ALPHA,
                             k_neighbors=K_NEIGHBORS,
                             density_percentile=DENSITY_PERCENTILE,
                             combine_rule=COMBINE_RULE,
                             isolated_density_factor=ISOLATED_DENSITY_FACTOR,
                             iterative=ITERATIVE,
                             max_iter=MAX_ITER):
    pts = points.copy()
    n_total = pts.shape[0]
    # result mask (True = keep)
    global_keep = np.ones(n_total, dtype=bool)

    # choose K
    if user_k is not None:
        k = int(user_k)
    else:
        # simple elbow-like heuristic: try 1..max_k_try and pick largest relative drop
        inertias = []
        ks = list(range(1, min(max_k_try, max(1, n_total-1)) + 1))
        for kk in ks:
            if SKLEARN:
                km = KMeans(n_clusters=kk, init='k-means++', n_init=10, random_state=1)
                km.fit(pts)
                inertias.append(km.inertia_)
            else:
                # use simple kmeans (small scale)
                # re-use sklearn-free quick approximate: randomly init centers then Lloyd a few iters
                # (for brevity implement a cheap version)
                centers = pts[np.random.choice(n_total, kk, replace=False)]
                for _ in range(50):
                    labels = np.argmin(((pts[:,None,:] - centers[None,:,:])**2).sum(axis=2), axis=1)
                    new_centers = np.array([pts[labels==j].mean(axis=0) if np.any(labels==j) else centers[j] for j in range(kk)])
                    if np.allclose(new_centers, centers, atol=1e-6): break
                    centers = new_centers
                inertias.append(((pts - centers[labels])**2).sum())
        if len(inertias) <= 1:
            k = 1
        else:
            drops = [(inertias[i] - inertias[i+1]) / inertias[i] if inertias[i] > 0 else 0 for i in range(len(inertias)-1)]
            argmax = int(np.argmax(drops))
            k = argmax + 2
    if VERBOSE:
        print(f"[info] chosen k = {k}")

    # main iterative loop: recluster after removals
    for iteration in range(max_iter if iterative else 1):
        if VERBOSE:
            print(f"[iter {iteration+1}] points remaining: {global_keep.sum()} / {n_total}")
        # points to cluster this iteration
        idxs = np.where(global_keep)[0]
        if idxs.size == 0:
            break
        data = pts[idxs]

        # cluster
        if SKLEARN:
            if k <= 0:
                labels = np.zeros(data.shape[0], dtype=int)
            else:
                model = KMeans(n_clusters=min(k, max(1, data.shape[0])), init='k-means++', n_init=10, random_state=2)
                labels = model.fit_predict(data)
        else:
            # fallback simple kmeans (same as above)
            kk = min(k, max(1, data.shape[0]))
            centers = data[np.random.choice(data.shape[0], kk, replace=False)]
            for _ in range(100):
                l = np.argmin(((data[:,None,:] - centers[None,:,:])**2).sum(axis=2), axis=1)
                new_centers = np.array([data[l==j].mean(axis=0) if np.any(l==j) else centers[j] for j in range(kk)])
                if np.allclose(new_centers, centers, atol=1e-6): break
                centers = new_centers
            labels = l

        # compute global density scores once (for points considered)
        density_scores = compute_local_density(data, k=k_neighbors)

        # per-cluster decisions
        to_remove = np.zeros(data.shape[0], dtype=bool)
        for c in np.unique(labels):
            cluster_idx_local = np.where(labels == c)[0]
            if cluster_idx_local.size < MIN_CLUSTER_SIZE:
                # skip cluster-level removal; but isolated points still may be removed by density rule below
                continue
            cluster_pts = data[cluster_idx_local]
            mu, principal = pca_principal(cluster_pts)
            dists = perp_distances(cluster_pts, mu, principal)  # perp distances for points in this cluster

            # robust thresholding for perp distances
            if perp_method == 'mad':
                perp_thresh = robust_threshold_by_mad(dists, k=mad_k)
            else:
                # mean/std
                perp_thresh = dists.mean() + mean_std_alpha * dists.std(ddof=0)

            # map back to data-level arrays
            # density for cluster local points:
            dens_local = density_scores[cluster_idx_local]
            # thresholds for density: use percentile on whole data-> low density if below percentile
            density_thresh = np.percentile(density_scores, density_percentile)

            # isolation threshold
            isolated_thresh = np.median(density_scores) * isolated_density_factor

            # decide
            for local_pos, global_pos_in_data in enumerate(cluster_idx_local):
                perp = dists[local_pos]
                dens = dens_local[local_pos]
                # condition flags
                cond_perp = perp > perp_thresh
                cond_low_density = dens <= density_thresh
                cond_isolated = dens <= isolated_thresh
                if combine_rule == 'both':
                    out_flag = (cond_perp and cond_low_density) or cond_isolated
                else:  # 'either'
                    out_flag = cond_perp or cond_low_density or cond_isolated
                if out_flag:
                    to_remove[global_pos_in_data] = True

        # additionally apply global low-density-only removal for points not in clusters (or small clusters)
        # optional: remove bottom X% density points forcibly (very strict)
        # Here we already remove isolated ones by isolated_thresh above.

        # map to global_keep
        removed_idxs = idxs[to_remove]
        if VERBOSE:
            print(f"[iter {iteration+1}] marking {removed_idxs.size} points for removal")
        if removed_idxs.size == 0:
            break
        # update global mask
        global_keep[removed_idxs] = False

    # final: build report arrays
    kept = pts[global_keep]
    outliers = pts[~global_keep]
    # produce per-point report
    # recompute final clustering + distances for reporting simplicity
    idxs_final = np.where(global_keep)[0]
    final_data = pts[idxs_final]
    # compute final principal/distance at cluster-level for report
    report_records = []
    # cluster final for reporting:
    if final_data.shape[0] > 0:
        if SKLEARN:
            model = KMeans(n_clusters=min(k, max(1, final_data.shape[0])), init='k-means++', n_init=10, random_state=3)
            final_labels = model.fit_predict(final_data)
        else:
            # simple kmeans
            kk = min(k, max(1, final_data.shape[0]))
            centers = final_data[np.random.choice(final_data.shape[0], kk, replace=False)]
            for _ in range(100):
                l = np.argmin(((final_data[:,None,:] - centers[None,:,:])**2).sum(axis=2), axis=1)
                new_centers = np.array([final_data[l==j].mean(axis=0) if np.any(l==j) else centers[j] for j in range(kk)])
                if np.allclose(new_centers, centers, atol=1e-6): break
                centers = new_centers
            final_labels = l
    else:
        final_labels = np.array([], dtype=int)

    # for simplicity report original indices (index in original array)
    for orig_idx in range(n_total):
        rec = {'orig_index': int(orig_idx), 'X': float(points[orig_idx,0]), 'Y': float(points[orig_idx,1]), 'Z': float(points[orig_idx,2]), 'kept': bool(global_keep[orig_idx])}
        report_records.append(rec)

    report_df = pd.DataFrame(report_records).sort_values('orig_index').reset_index(drop=True)

    return global_keep, report_df, kept, outliers

# main

In [ ]:


def main():
    infile = find_input_file(INPUT_CSV)
    pts, df_in = load_points(infile)
    if VERBOSE:
        print("Loaded points:", pts.shape[0], "from", infile)
    keep_mask, report_df, kept_pts, outliers_pts = detect_outliers_pipeline(pts)

    # save outputs
    pd.DataFrame(kept_pts, columns=['X','Y','Z']).to_csv(OUTPUT_KEPT, index=False)
    pd.DataFrame(outliers_pts, columns=['X','Y','Z']).to_csv(OUTPUT_OUTLIERS, index=False)
    report_df.to_csv(OUTPUT_REPORT, index=False)
    if VERBOSE:
        print("Kept:", kept_pts.shape[0], "Outliers:", outliers_pts.shape[0])
        print("Saved files:", OUTPUT_KEPT, OUTPUT_OUTLIERS, OUTPUT_REPORT)

    # optional plot
    if PLOTTING:
        fig = plt.figure(figsize=(10,7))
        ax = fig.add_subplot(111, projection='3d')
        if kept_pts.shape[0] > 0:
            ax.scatter(kept_pts[:,0], kept_pts[:,1], kept_pts[:,2], s=20, label='kept')
        if outliers_pts.shape[0] > 0:
            ax.scatter(outliers_pts[:,0], outliers_pts[:,1], outliers_pts[:,2], c='r', s=40, label='outliers')
        ax.set_title('Denoising result (PCPCA + density)')
        ax.legend()
        plt.show()

if __name__ == '__main__':
    main()
 